# Generating non-coding genome coordinates

In [ ]:
## extract exons, introns, UTRs and intergenic regions
# gtftools -e /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.exons.bed /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.gtf
# gtftools -i /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.introns.bed /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.gtf
# gtftools -u /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.utr.bed /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.gtf
# gtftools -b /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.intergenic.bed /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.gtf



In [ ]:
# sort bedfiles by chrom and then coord

bedtools sort -i /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.introns.bed > /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.introns.sorted.bed
bedtools sort -i /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.utr.bed > /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.utr.sorted.bed
bedtools sort -i /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.intergenic.bed > /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.intergenic.sorted.bed

In [1]:
! agat_sp_extract_attributes.pl --gff /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.gtf  -att locus_tag,product,name -p level3,CDS -o /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.cds.gtf

/bin/bash: line 1: agat_sp_extract_attributes.pl: command not found


In [ ]:
cat /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.introns.sorted.bed /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.utr.sorted.bed /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.intergenic.sorted.bed \
| bedtools sort -i - | bedtools merge -i - > /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.noncoding.bed

In [ ]:
GTF=/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.gtf
GENOME=hg38.chrom.sizes   # chrom\tlength

# 1. CDS intervals (GTF is 1-based inclusive → BED is 0-based half-open)
awk 'BEGIN{OFS="\t"} $3=="CDS" {print $1,$4-1,$5}' /users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.gtf \
  | bedtools sort -i - | bedtools merge -i - > /users/PAS2905/coraalbers/ag/ag_data/cds.bed

cut -f1,2 /users/PAS2905/coraalbers/ag/hg38.fa.fai > hg38.chrom.sizes

# 2. Noncoding = genome complement of CDS
bedtools complement -i /users/PAS2905/coraalbers/ag/ag_data/cds.bed -g /users/PAS2905/coraalbers/ag/hg38.chrom.sizes > /users/PAS2905/coraalbers/ag/ag_data/noncoding.bed

# 3. remove chr annotation to match vcf file chromosome naming
sed 's/^chr//' noncoding.bed > noncoding_num_chr.bed